# **Implementación de un Modelo de Machine Learning Aplicado al Puesto de Trabajo**
## Predicción de mora en créditos de motocicletas – Motocity El Salvador

**Universidad Francisco Gavidia** – Maestría en Gestión Estratégica de Tecnologías de la Información
**Asignatura:** Inteligencia Artificial – Ciclo 02-2026
**Docente:** Dr. Edwin Alberto Callejas
**Estudiante:** Victor Manuel Hernández Hernández
**Carnet:** ____________

---

**Pregunta de investigación:** ¿Cómo puede un modelo de Machine Learning ayudar a resolver, predecir o mejorar un problema real relacionado con las actividades de mi puesto de trabajo, y qué nivel de efectividad puede alcanzar utilizando datos recopilados y procesados por mí mismo?

En mi caso, quiero saber si un modelo puede **avisar, al momento de aprobar un crédito de moto, si ese cliente tiene riesgo de caer en mora**.

# **1. Definición del problema**

### Contexto organizacional
Trabajo como Ingeniero de Desarrollo e Infraestructura en **Motocity El Salvador**, empresa que vende motocicletas de las marcas Katana y Dayun en más de 20 sucursales del país. Una gran parte de las motos se venden **al crédito**: el cliente da una prima (desde el 15 %) y paga el resto en cuotas de hasta 48 meses. Los datos de estos créditos se registran en SAP Business One, que es el sistema con el que trabajo todos los días.

### Necesidad o problema
Algunos clientes dejan de pagar sus cuotas y **caen en mora**. Cuando eso pasa, la empresa pierde dinero, tiene que hacer gestiones de cobro y a veces recuperar la moto. Hoy la decisión de aprobar un crédito depende mucho de la experiencia del analista, y no hay una herramienta que diga de forma rápida qué tan riesgoso es un cliente.

### ¿Por qué Machine Learning?
- Hay **datos históricos** de créditos anteriores donde ya se sabe quién pagó y quién cayó en mora. Un modelo puede aprender de esos casos.
- La mora no depende de una sola cosa, sino de una **combinación** de factores (prima, plazo, ingreso, tipo de trabajo, etc.). Es difícil verlo a simple vista, pero un modelo sí puede encontrar esos patrones.
- Una vez entrenado, el modelo da una respuesta **en segundos** para cada solicitud nueva.

### Variable objetivo (target)
**`U_Mora`**: indica si el cliente cayó en mora (más de 60 días de atraso en alguna cuota).
- `1` = sí cayó en mora
- `0` = pagó con normalidad

Como la respuesta es Sí/No, este es un problema de **clasificación**.

# **2. Construcción del Dataset**

### Origen de los datos
Por **confidencialidad**, no puedo usar ni subir a Colab datos reales de clientes de la empresa. Por eso construí un **dataset simulado**, pero basado en información real del negocio:

- La **estructura** imita la forma en que SAP Business One guarda los documentos (`DocEntry`, `CardCode`, `DocDate` y campos de usuario que empiezan con `U_`).
- Las **sucursales** y los **modelos de moto** son los que aparecen en el sitio web oficial www.motocity.com.sv.
- Las **reglas de crédito** también salen de la página de financiamiento: prima mínima del 15 %, plazos de hasta 48 meses, ingreso mínimo de $300 y tipos de ingreso (empleo formal, negocio propio o remesas).
- Los **precios** son aproximados según el cilindraje de cada moto.
- Los **nombres, DUI y teléfonos no se incluyen**: cada cliente solo tiene un código ficticio.

El dataset lo generé con un script de Python y lo subí a mi repositorio de GitHub, desde donde lo cargo en este notebook. Al generarlo **agregué a propósito algunos errores** comunes en la vida real (datos vacíos, registros repetidos, edades imposibles, etc.) para poder practicar la limpieza.

### Diccionario de variables

| Variable | Tipo | Descripción |
|---|---|---|
| `DocEntry` | Número | Número interno del crédito (como en SAP) |
| `CardCode` | Texto | Código del cliente (ficticio) |
| `DocDate` | Fecha | Fecha en que se otorgó el crédito |
| `U_Sucursal` | Texto | Sucursal donde se vendió la moto |
| `U_Marca` | Texto | Marca de la moto (Katana o Dayun) |
| `U_Modelo` | Texto | Modelo de la moto |
| `U_Cilindraje` | Número | Cilindraje del motor (cc) |
| `U_PrecioMoto` | Número | Precio de la moto en dólares |
| `U_Prima` | Número | Prima que pagó el cliente (dólares) |
| `U_Plazo` | Número | Plazo del crédito en meses (12, 18, 24, 36 o 48) |
| `U_Cuota` | Número | Cuota mensual en dólares |
| `U_Edad` | Número | Edad del cliente |
| `U_TipoIngreso` | Texto | De dónde vienen sus ingresos: Empleo formal, Negocio propio o Remesas |
| `U_IngresoMensual` | Número | Ingreso mensual aproximado del cliente |
| `U_AniosTrabajo` | Número | Años en su trabajo o negocio actual |
| `U_Dependientes` | Número | Personas que dependen económicamente del cliente |
| `U_Comprobantes` | Sí/No | Si presentó comprobantes de ingreso |
| `U_CreditoPrevio` | Sí/No | Si ya había tenido un crédito antes |
| **`U_Mora`** | **0/1** | **Variable objetivo: 1 = cayó en mora** |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Cargar el dataset desde mi repositorio de GitHub
url = "https://raw.githubusercontent.com/Victormh96/ml-mora-motocity/main/creditos_motocity.csv"
df_original = pd.read_csv(url)

print("Filas y columnas:", df_original.shape)
df_original.head()

In [ ]:
# Información general: tipo de dato y cuántos valores tiene cada columna
df_original.info()

In [ ]:
# Resumen estadístico de las columnas numéricas
df_original.describe().round(2)

**¿Qué se ve aquí?** El dataset tiene **1,012 registros** y **19 columnas**. Desde ya se notan algunos problemas: hay columnas con menos valores que el total (o sea, datos vacíos), y en la edad aparece un mínimo de **0** y un máximo de **210** años, lo cual es imposible. Eso lo arreglo en la siguiente sección.

# **3. Limpieza de Datos**

Antes de entrenar cualquier modelo hay que revisar la calidad de los datos. Voy a revisar uno por uno los problemas y explicar qué hice con cada uno.

In [ ]:
# Trabajo sobre una copia para no tocar los datos originales
df = df_original.copy()

# 3.1 Datos duplicados (el mismo crédito guardado dos veces)
print("Registros duplicados:", df.duplicated().sum())
df = df.drop_duplicates()
print("Registros después de quitar duplicados:", len(df))

In [ ]:
# 3.2 Valores nulos (vacíos) por columna
nulos = df.isnull().sum()
nulos[nulos > 0]

In [ ]:
# 3.3 Errores de escritura en sucursal y marca
print("Sucursales distintas ANTES de corregir:", df["U_Sucursal"].nunique())
print(sorted(df["U_Sucursal"].dropna().unique()))

In [ ]:
# Quito espacios, uso el mismo formato de mayúsculas y corrijo abreviaturas
df["U_Sucursal"] = df["U_Sucursal"].str.strip().str.title().replace({"Sta. Tecla": "Santa Tecla"})
df["U_Marca"] = df["U_Marca"].str.upper()

print("Sucursales distintas DESPUÉS de corregir:", df["U_Sucursal"].nunique())
print("Marcas:", df["U_Marca"].unique())

In [ ]:
# 3.4 Tratamiento de nulos
# Sucursal vacía: no se puede adivinar, la marco como "Sin Dato"
df["U_Sucursal"] = df["U_Sucursal"].fillna("Sin Dato")

# Años de trabajo vacío: lo lleno con la mediana (el valor del medio)
df["U_AniosTrabajo"] = df["U_AniosTrabajo"].fillna(df["U_AniosTrabajo"].median())

In [ ]:
# 3.5 Valores atípicos en el ingreso
plt.figure(figsize=(10, 3))
sns.boxplot(x=df["U_IngresoMensual"])
plt.title("Ingreso mensual (antes de corregir)")
plt.show()

print(df.loc[df["U_IngresoMensual"] > 5000, ["CardCode", "U_TipoIngreso", "U_IngresoMensual"]])

In [ ]:
# Ingresos de más de $5,000 al mes no son normales para un cliente que compra moto al crédito.
# Revisándolos, parece que se digitaron con dos ceros de más, así que los divido entre 100.
df.loc[df["U_IngresoMensual"] > 5000, "U_IngresoMensual"] = df["U_IngresoMensual"] / 100

# Ingreso vacío: lo lleno con la mediana según el tipo de ingreso
# (no gana lo mismo alguien que recibe remesas que alguien con empleo formal)
df["U_IngresoMensual"] = df["U_IngresoMensual"].fillna(
    df.groupby("U_TipoIngreso")["U_IngresoMensual"].transform("median"))

In [ ]:
# 3.6 Edades imposibles (menores de 18 o mayores de 80)
edad_mal = ~df["U_Edad"].between(18, 80)
print("Edades fuera de rango:", df.loc[edad_mal, "U_Edad"].tolist())

mediana_edad = df.loc[~edad_mal, "U_Edad"].median()
df.loc[edad_mal, "U_Edad"] = mediana_edad
print("Se reemplazaron por la mediana:", mediana_edad)

In [ ]:
# 3.7 Primas negativas (una prima no puede ser negativa, es un error de signo)
print("Primas negativas:", (df["U_Prima"] < 0).sum())
df["U_Prima"] = df["U_Prima"].abs()

# Verificación final: ya no deben quedar nulos
print("Nulos restantes:", df.isnull().sum().sum())
print("Registros finales:", len(df))

### Resumen de la limpieza

| Problema encontrado | Registros afectados | Tratamiento aplicado |
|---|---|---|
| Créditos duplicados | 12 | Se eliminaron, dejando solo uno de cada uno |
| Sucursal vacía | 6 | Se marcó como "Sin Dato" |
| Sucursal mal escrita (mayúsculas, espacios, "Sta. Tecla") | 32 nombres distintos → 24 | Se unificó el formato |
| Marca escrita diferente ("Katana" y "KATANA") | 4 valores → 2 | Se pasó todo a mayúsculas |
| Ingreso mensual vacío | 18 | Se llenó con la mediana según el tipo de ingreso |
| Ingresos exagerados (más de $5,000) | 4 | Se corrigieron dividiendo entre 100 (error de digitación) |
| Años de trabajo vacío | 11 | Se llenó con la mediana (3.4 años) |
| Edades imposibles (0, 3, 9, 150, 210) | 5 | Se reemplazaron por la mediana (33 años) |
| Primas negativas | 3 | Se corrigió el signo |

Al final quedaron **1,000 créditos limpios** y sin datos vacíos.

No eliminé las filas con datos vacíos porque eran pocas y cada crédito tiene información útil en las demás columnas. Preferí completarlas con la mediana, que no se ve afectada por los valores extremos.

# **4. Preprocesamiento**

En esta parte preparo los datos para que el modelo los pueda entender.

In [ ]:
# Primero veo cuántos clientes cayeron en mora
conteo = df["U_Mora"].value_counts()
porcentaje = df["U_Mora"].value_counts(normalize=True) * 100
print(pd.DataFrame({"Créditos": conteo, "Porcentaje (%)": porcentaje.round(1)}))

sns.countplot(x="U_Mora", data=df, hue="U_Mora", palette=["#4C9F70", "#D1495B"], legend=False)
plt.xticks([0, 1], ["Pagó normal (0)", "Cayó en mora (1)"])
plt.title("Distribución de la variable objetivo")
plt.ylabel("Cantidad de créditos")
plt.xlabel("")
plt.show()

Solo el **20.7 %** de los clientes cayó en mora. Esto se llama **desbalance de clases**: hay muchos más clientes buenos que morosos. Es normal en créditos, pero puede hacer que el modelo "aprenda" a decir casi siempre que el cliente va a pagar. Lo tomo en cuenta más adelante, en la segunda iteración.

### 4.1 Ingeniería de características (crear variables nuevas)
Por mi experiencia en el negocio, sé que no importa tanto la cuota sola, sino **cuánto representa del ingreso** del cliente. Por eso creo tres variables nuevas:

- **`PrimaPct`**: qué porcentaje del precio pagó de prima.
- **`CargaCuota`**: qué parte de su ingreso se va en la cuota (por ejemplo, 0.25 = la cuota es la cuarta parte de su sueldo).
- **`MontoFinanciado`**: cuánto dinero se le prestó realmente (precio − prima).

In [ ]:
df["PrimaPct"] = df["U_Prima"] / df["U_PrecioMoto"]
df["CargaCuota"] = df["U_Cuota"] / df["U_IngresoMensual"]
df["MontoFinanciado"] = df["U_PrecioMoto"] - df["U_Prima"]

# ¿Se nota diferencia entre morosos y no morosos?
df.groupby("U_Mora")[["PrimaPct", "CargaCuota", "U_Plazo", "U_IngresoMensual"]].mean().round(3)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(x="U_Mora", y="CargaCuota", data=df, hue="U_Mora", palette=["#4C9F70", "#D1495B"], legend=False, ax=ax[0])
ax[0].set_title("Carga de la cuota sobre el ingreso")
ax[0].set_xticks([0, 1], ["Pagó normal", "Mora"])
ax[0].set_xlabel("")
sns.boxplot(x="U_Mora", y="PrimaPct", data=df, hue="U_Mora", palette=["#4C9F70", "#D1495B"], legend=False, ax=ax[1])
ax[1].set_title("Porcentaje de prima")
ax[1].set_xticks([0, 1], ["Pagó normal", "Mora"])
ax[1].set_xlabel("")
plt.tight_layout()
plt.show()

Se ve claro que los clientes que cayeron en mora tienen una **cuota más pesada en relación con su ingreso** y pagaron **menos prima**. Esto confirma que las variables nuevas tienen sentido.

### 4.2 Selección de variables
Quito las columnas que no deben usarse para predecir:
- `DocEntry`, `CardCode` y `DocDate`: son solo identificadores o fechas, no dicen nada del cliente.
- `U_Modelo`: hay muchos modelos y ya tengo el precio y el cilindraje, que resumen esa información.
- `U_Sucursal`: decidí no usarla para que el modelo evalúe al cliente por su **situación económica** y no por la zona donde compró. Me parece más justo.

### 4.3 Codificación de variables categóricas
Los modelos solo entienden números, así que convierto las columnas de texto (marca, tipo de ingreso, comprobantes y crédito previo) en columnas de 0 y 1 con **One-Hot Encoding**.

In [ ]:
X = df.drop(columns=["DocEntry", "CardCode", "DocDate", "U_Sucursal", "U_Modelo", "U_Mora"])
X = pd.get_dummies(X, columns=["U_Marca", "U_TipoIngreso", "U_Comprobantes", "U_CreditoPrevio"],
                   drop_first=True).astype(float)
y = df["U_Mora"]

print("Variables que usará el modelo:", X.shape[1])
print(list(X.columns))

### 4.4 División en entrenamiento y prueba
Separo los datos: **80 %** para que el modelo aprenda y **20 %** para probarlo con casos que nunca ha visto. Uso `stratify=y` para que en ambos grupos quede el mismo porcentaje de morosos.

### 4.5 Escalado
Las variables tienen escalas muy distintas (el precio está en miles y la carga de la cuota en decimales). Con **StandardScaler** las pongo todas en una escala parecida para que ninguna pese más solo por ser un número más grande.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

escalador = StandardScaler()
X_train_esc = escalador.fit_transform(X_train)   # aprende la escala solo con entrenamiento
X_test_esc = escalador.transform(X_test)         # y la aplica a prueba

print("Entrenamiento:", X_train.shape[0], "créditos")
print("Prueba:", X_test.shape[0], "créditos")
print("Morosos en prueba:", y_test.sum())

**Balanceo de clases:** en esta primera versión **no** balanceo los datos, para ver cómo se comporta el modelo "normal". El balanceo lo aplico como mejora en la **sección 8**.

# **5. Implementación del Modelo**

Voy a entrenar dos modelos:

1. **Regresión Logística** – la uso como **punto de comparación** (línea base). Es un modelo sencillo que calcula la probabilidad de que algo pase, y es muy usado en bancos para evaluar créditos.
2. **Random Forest** – es mi **modelo principal**.

### ¿Por qué elegí Random Forest?
- Funciona como un **grupo de muchos árboles de decisión** que "votan". Cada árbol hace preguntas tipo *"¿la prima es menor al 18 %?"* o *"¿tiene comprobantes?"*, y la respuesta final es la que elige la mayoría.
- Funciona bien con datos de tabla como los de SAP, mezclando números y categorías.
- Encuentra **combinaciones** de factores (por ejemplo, poca prima *y* plazo largo).
- Me dice **qué variables pesaron más** en la decisión, lo que sirve para explicarle el resultado al área de créditos.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Modelo 1: Regresión Logística (línea base)
modelo_lr = LogisticRegression(max_iter=1000)
modelo_lr.fit(X_train_esc, y_train)
pred_lr = modelo_lr.predict(X_test_esc)

# Modelo 2: Random Forest (modelo principal)
modelo_rf = RandomForestClassifier(n_estimators=300, random_state=42)
modelo_rf.fit(X_train_esc, y_train)
pred_rf = modelo_rf.predict(X_test_esc)

print("Modelos entrenados")

# **6. Evaluación del Modelo**

Antes de ver los números, explico qué significa cada métrica con el ejemplo de los créditos:

| Métrica | ¿Qué responde? |
|---|---|
| **Accuracy (exactitud)** | De todos los clientes, ¿a cuántos les atinó el modelo? |
| **Precision (precisión)** | De los clientes que el modelo marcó como "mora", ¿cuántos de verdad cayeron en mora? |
| **Recall (sensibilidad)** | De todos los clientes que de verdad cayeron en mora, ¿a cuántos detectó el modelo? |
| **F1-Score** | Un promedio entre precision y recall |

Para Motocity la métrica **más importante es el Recall**, porque lo que más cuesta es **no detectar** a un cliente que después no paga. Una falsa alarma solo significa revisar mejor el caso o pedir más prima.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

def metricas(y_real, y_pred):
    return {
        "Accuracy (%)": round(accuracy_score(y_real, y_pred) * 100, 1),
        "Precision (%)": round(precision_score(y_real, y_pred, zero_division=0) * 100, 1),
        "Recall (%)": round(recall_score(y_real, y_pred) * 100, 1),
        "F1-Score (%)": round(f1_score(y_real, y_pred) * 100, 1),
    }

resultados_1 = pd.DataFrame({
    "Regresión Logística": metricas(y_test, pred_lr),
    "Random Forest": metricas(y_test, pred_rf),
}).T
resultados_1

In [ ]:
print("Reporte detallado - Random Forest")
print(classification_report(y_test, pred_rf, target_names=["Pagó normal", "Mora"]))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for i, (nombre, pred) in enumerate([("Regresión Logística", pred_lr), ("Random Forest", pred_rf)]):
    sns.heatmap(confusion_matrix(y_test, pred), annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax[i],
                xticklabels=["Pagó normal", "Mora"], yticklabels=["Pagó normal", "Mora"], annot_kws={"size": 14})
    ax[i].set_title(nombre)
    ax[i].set_xlabel("Lo que predijo el modelo")
    ax[i].set_ylabel("Lo que pasó en realidad")
plt.tight_layout()
plt.show()

### ¿Qué significan estos resultados?

- El **Random Forest** tiene un accuracy de **86.5 %**, que suena muy bien. Pero hay que tener cuidado: como solo el 21 % de los clientes cae en mora, un modelo que dijera *"todos van a pagar"* ya tendría casi 80 % de exactitud sin hacer nada.
- Su **precision es de 93.8 %**: cuando el modelo dice "este cliente va a caer en mora", casi siempre tiene razón.
- El problema está en el **recall de 36.6 %**. De los **41 clientes morosos** del grupo de prueba, solo detectó **15** y se le escaparon **26** (la casilla de abajo a la izquierda en la matriz).
- La **Regresión Logística** se comporta parecido: recall de **41.5 %**.

**Conclusión de esta primera versión:** los modelos son muy "confiados" y casi siempre dicen que el cliente va a pagar, por el desbalance de clases. Para el negocio, dejar pasar a más de la mitad de los morosos no es aceptable. Esto lo trabajo en la sección 8.

# **7. Interpretación de Resultados**

In [ ]:
# ¿Qué variables usó más el Random Forest para decidir?
importancia = pd.Series(modelo_rf.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(x=importancia.head(10).values * 100, y=importancia.head(10).index, color="#2E86AB")
plt.title("Las 10 variables más importantes para el modelo")
plt.xlabel("Importancia (%)")
plt.ylabel("")
plt.tight_layout()
plt.show()

(importancia.head(10) * 100).round(1)

In [ ]:
# Para confirmar la dirección (si aumenta o disminuye el riesgo) reviso los coeficientes de la Regresión Logística
coef = pd.Series(modelo_lr.coef_[0], index=X.columns).sort_values()
plt.figure(figsize=(10, 5))
colores = ["#4C9F70" if c < 0 else "#D1495B" for c in coef]
plt.barh(coef.index, coef.values, color=colores)
plt.axvline(0, color="gray", linewidth=0.8)
plt.title("Rojo = aumenta el riesgo de mora   |   Verde = lo disminuye")
plt.tight_layout()
plt.show()

### ¿Qué tan efectivo resultó el modelo?
En esta primera versión, el modelo es **bueno para no dar falsas alarmas** (precision de 93.8 %), pero **débil para detectar morosos** (recall de 36.6 %). Sirve como punto de partida, pero todavía no está listo para usarse.

### ¿Qué variables tienen mayor influencia?
Las que más pesan son **la carga de la cuota sobre el ingreso, el ingreso mensual, el valor de la cuota, el monto financiado**. Esto tiene mucho sentido en el negocio:
- Si la cuota se lleva una parte grande del sueldo, cualquier gasto extra hace que el cliente se atrase.
- La gráfica de coeficientes confirma que **no tener comprobantes, vivir de remesas, poca prima y plazos largos aumentan el riesgo**, mientras que **haber tenido crédito antes y más años en el trabajo lo reducen**.

### ¿Es viable implementarlo en la organización?
Técnicamente **sí**. Los datos que usa el modelo ya existen en SAP Business One y en el formulario de solicitud de crédito, y en MotoCity Go ya tenemos la integración con SAP. El modelo podría mostrar un "semáforo de riesgo" al analista cuando revisa una solicitud. Pero primero hay que mejorar el recall (sección 8).

### ¿Qué beneficios aportaría al negocio?
- **Menos pérdidas** por créditos que no se pagan.
- **Decisiones más rápidas y parejas** entre sucursales, sin depender solo de la experiencia de cada analista.
- **Mejores condiciones para el cliente**: en vez de rechazar, se puede ofrecer más prima o un plazo más corto a quien tiene riesgo alto.
- Ayuda a enfocar el **seguimiento de cobro** desde el inicio en los clientes con más riesgo.

# **8. Optimización y Segunda Iteración**

### ¿Qué cambié?
El problema principal de la primera versión fue el **desbalance de clases**. Por eso hice dos mejoras juntas:

1. **Balanceo de clases** con `class_weight="balanced"`: le digo al modelo que **equivocarse con un moroso cuesta más** que equivocarse con un cliente bueno. Así el modelo le pone más atención a los casos de mora.
2. **Ajuste de hiperparámetros** del Random Forest:
   - `max_depth=6`: limito qué tan profundo crece cada árbol para que no se "memorice" los datos de entrenamiento.
   - `min_samples_leaf=5`: cada decisión final debe basarse en al menos 5 clientes y no en casos aislados.

In [ ]:
# Regresión Logística balanceada
modelo_lr2 = LogisticRegression(max_iter=1000, class_weight="balanced")
modelo_lr2.fit(X_train_esc, y_train)
pred_lr2 = modelo_lr2.predict(X_test_esc)

# Random Forest balanceado y ajustado
modelo_rf2 = RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_leaf=5,
                                    class_weight="balanced", random_state=42)
modelo_rf2.fit(X_train_esc, y_train)
pred_rf2 = modelo_rf2.predict(X_test_esc)

comparacion = pd.DataFrame({
    "1ª iteración - Regresión Logística": metricas(y_test, pred_lr),
    "1ª iteración - Random Forest": metricas(y_test, pred_rf),
    "2ª iteración - Regresión Logística balanceada": metricas(y_test, pred_lr2),
    "2ª iteración - Random Forest balanceado": metricas(y_test, pred_rf2),
}).T
comparacion

In [ ]:
# Comparación visual del Random Forest antes y después
comp_rf = comparacion.loc[["1ª iteración - Random Forest", "2ª iteración - Random Forest balanceado"]].T
comp_rf.columns = ["Antes", "Después"]
comp_rf.plot(kind="bar", figsize=(10, 4.5), color=["#B0B7C3", "#2E86AB"], rot=0)
plt.title("Random Forest: antes y después de la mejora")
plt.ylabel("%")
plt.ylim(0, 100)
plt.legend(loc="upper right")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for i, (nombre, pred) in enumerate([("Antes (1ª iteración)", pred_rf), ("Después (2ª iteración)", pred_rf2)]):
    sns.heatmap(confusion_matrix(y_test, pred), annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax[i],
                xticklabels=["Pagó normal", "Mora"], yticklabels=["Pagó normal", "Mora"], annot_kws={"size": 14})
    ax[i].set_title("Random Forest - " + nombre)
    ax[i].set_xlabel("Lo que predijo el modelo")
    ax[i].set_ylabel("Lo que pasó en realidad")
plt.tight_layout()
plt.show()

### ¿Cómo impactó la mejora?

| Random Forest | Antes | Después |
|---|---|---|
| Accuracy | 86.5 % | 82.0 % |
| Precision | 93.8 % | 55.8 % |
| **Recall** | **36.6 %** | **58.5 %** |
| F1-Score | 52.6 % | 57.1 % |
| Morosos detectados | 15 de 41 | 24 de 41 |

- El **recall subió de 36.6 % a 58.5 %**: ahora el modelo detecta **24 morosos** en lugar de 15. Esa era la mejora que buscaba.
- El **F1-Score también subió** (52.6 % → 57.1 %), o sea que en balance el modelo quedó mejor.
- A cambio, la **precision bajó bastante** (93.8 % → 55.8 %) y el accuracy un poco (86.5 % → 82.0 %). Esto pasa porque ahora el modelo da más falsas alarmas: 19 clientes buenos marcados como riesgosos, en vez de 1. Para Motocity el cambio conviene, porque **revisar un caso de más o pedir más prima cuesta mucho menos que perder un crédito**.
- La **Regresión Logística balanceada** detectó todavía más morosos (recall de 68.3 %), pero con más falsas alarmas (29). Las dos opciones son válidas; el Random Forest balanceado queda más **equilibrado**.

# **9. Conclusiones**

### ¿Se cumplió el objetivo planteado?
**Sí.** Logré construir un modelo que, con datos que normalmente ya existen en SAP, puede identificar a una parte importante de los clientes con riesgo de mora. Después de la segunda iteración, el Random Forest detecta el **58.5 %** de los morosos, frente al 36.6 % de la primera versión. No es perfecto, pero ya sería una ayuda real para el analista de créditos.

### ¿Qué limitaciones tuvo el proyecto?
- **Los datos son simulados.** Aunque se basan en reglas reales del negocio, el resultado con datos reales podría ser distinto. Es la limitación más importante.
- **Pocos datos**: 1,000 créditos, y en el grupo de prueba solo hay 41 morosos, así que unos pocos aciertos o errores cambian bastante los porcentajes.
- **Faltan variables** que en la vida real influyen mucho: la puntuación en el buró de crédito, las deudas que ya tiene el cliente o si cambió de trabajo durante el crédito.
- El modelo **no explica por qué** un cliente específico es riesgoso; solo da el resultado general.

### ¿Qué mejoras se podrían hacer en el futuro?
- Probarlo con **datos reales anonimizados** dentro de los servidores de la empresa, sin sacar la información a internet.
- Agregar **información del buró de crédito** y del historial de pagos.
- Ajustar el **umbral de decisión**: en lugar de solo "mora sí/no", dar un **puntaje de riesgo** (bajo, medio, alto) para que el analista decida.
- Probar otros algoritmos como **XGBoost** y usar **validación cruzada** para que los resultados sean más estables.
- **Integrarlo en MotoCity Go**, para que el resultado aparezca directamente cuando se ingresa una solicitud de crédito.

# **Preguntas de análisis**

**1. ¿Cuál fue el problema de negocio identificado y por qué es relevante para tu organización?**
El problema es que algunos clientes que compran su moto al crédito dejan de pagar y caen en mora. Es relevante porque gran parte de las ventas de Motocity son al crédito, y cada cliente que no paga significa dinero perdido, gastos de cobro y a veces tener que recuperar la moto. Si se pudiera saber el riesgo desde el inicio, se tomarían mejores decisiones al aprobar.

**2. ¿Cómo construiste el dataset y qué desafíos encontraste durante la recopilación de datos?**
El mayor desafío fue la **confidencialidad**: no puedo sacar información real de clientes de la empresa ni subirla a Colab. Por eso construí un dataset simulado de 1,012 créditos con la misma estructura que usa SAP Business One. Para que fuera realista, tomé las sucursales y los modelos de moto del sitio web de Motocity, y las reglas de crédito de su página de financiamiento (prima mínima del 15 %, plazo de hasta 48 meses, ingreso mínimo de $300). Otro reto fue decidir qué variables incluir sin meter datos personales: dejé solo un código por cliente, sin nombre ni DUI.

**3. ¿Qué problemas de calidad de datos identificaste y cómo los solucionaste?**
Encontré 12 créditos duplicados (los eliminé); datos vacíos en sucursal, ingreso y años de trabajo (los llené con "Sin Dato" o con la mediana); sucursales y marcas escritas de diferentes formas (las unifiqué); ingresos exagerados por un error de digitación (los corregí); edades imposibles como 0 o 210 años (las reemplacé por la mediana); y primas negativas (corregí el signo). Al final quedaron 1,000 registros limpios.

**4. ¿Qué técnicas de preprocesamiento aplicaste y por qué fueron necesarias?**
- Creé variables nuevas como el **porcentaje de prima** y la **carga de la cuota sobre el ingreso**, porque por experiencia sé que importan más que los montos solos.
- Convertí las columnas de texto en números con **One-Hot Encoding**, porque los modelos solo trabajan con números.
- **Escalé** las variables para que ninguna pesara más solo por ser un número grande.
- **Dividí** los datos en 80 % para entrenar y 20 % para probar.
- En la segunda iteración **balanceé las clases**, porque solo el 21 % de los clientes cae en mora.

**5. ¿Por qué seleccionaste el modelo de Machine Learning utilizado?**
Elegí **Random Forest** porque combina muchos árboles de decisión, funciona bien con datos de tabla como los de SAP, encuentra combinaciones de factores y además dice qué variables fueron las más importantes, lo que ayuda a explicarle el resultado al área de créditos. Usé la **Regresión Logística** como punto de comparación, porque es el modelo tradicional en la evaluación de créditos.

**6. ¿Qué métricas utilizaste para evaluar el desempeño del modelo y qué significan sus valores?**
Usé accuracy, precision, recall, F1-Score y la matriz de confusión. En la versión final del Random Forest: el **accuracy de 82.0 %** indica que acierta en la mayoría de los casos; la **precision de 55.8 %** dice que, de los que marca como morosos, ese porcentaje sí lo es; y el **recall de 58.5 %** dice que detecta a más de la mitad de los morosos reales. Le di más importancia al recall porque lo más caro para la empresa es no detectar a un moroso.

**7. ¿Qué conclusiones puedes extraer a partir de los resultados obtenidos?**
Que la mora sí se puede anticipar con los datos del crédito. Lo que más pesa es **cuánto representa la cuota del ingreso del cliente** y **cuánta prima paga**, junto con tener comprobantes y experiencia previa con créditos. También aprendí que un accuracy alto puede engañar cuando hay pocos casos de mora: hay que mirar el recall.

**8. ¿Qué modificación realizaste en la segunda iteración y cómo impactó el desempeño del modelo?**
Apliqué **balanceo de clases** (`class_weight="balanced"`) y **ajusté hiperparámetros** del Random Forest (profundidad máxima de 6 y mínimo de 5 clientes por hoja). El recall subió de **36.6 % a 58.5 %** y el F1-Score de **52.6 % a 57.1 %**. El modelo pasó de detectar 15 a 24 de los 41 morosos. A cambio, la precision bajó de 93.8 % a 55.8 % porque aparecieron más falsas alarmas; para el negocio es un costo aceptable, ya que revisar un caso de más es mucho más barato que perder un crédito.

**9. ¿Recomendarías implementar la solución en tu organización? Justifica tu respuesta con evidencia.**
**Sí, pero como apoyo y no como decisión automática.** La evidencia es que el modelo mejorado detecta el 58.5 % de los morosos y, cuando marca a alguien, acierta en el 55.8 % de los casos. Lo recomendaría como un "semáforo de riesgo" en MotoCity Go que ayude al analista, quien toma la decisión final. Antes de usarlo en producción, habría que entrenarlo con datos reales de la empresa y hacer una prueba piloto en algunas sucursales.

**10. ¿Qué aprendiste durante el desarrollo de este proyecto y cómo podrías mejorar la solución en una siguiente versión?**
Aprendí que la mayor parte del trabajo está en **preparar bien los datos** y que conocer el negocio ayuda a crear mejores variables, como la carga de la cuota sobre el ingreso. También aprendí a no confiar solo en el accuracy y a elegir la métrica según lo que le importa a la empresa. En una siguiente versión usaría datos reales anonimizados, agregaría información del buró de crédito, daría un puntaje de riesgo en vez de un sí/no y lo conectaría a MotoCity Go para que se use en el día a día.